# 调试、版本控制与编程习惯

## 学习目标

- 掌握基本的调试方法（print → pdb → Jupyter调试器）
- 学会读懂常见Python报错信息
- 掌握Git基本操作（init, add, commit, push, pull）
- 建立良好的编程习惯（命名规范、注释、格式化）


## 1. 调试方法

调试是编程能力中最被忽视的，也是提升编程自信最有效的。

物理背景学生遇到报错时最常见的反应是"代码报错了"然后等待救援，而不是自己读报错信息、定位问题。

**目标**：建立"报错是朋友不是敌人"的心态——报错信息就是在告诉你哪里出了问题，学会读它就能解决大部分问题。

### 1.1 最常见的报错类型

在深度学习/音频处理中，最常遇到的报错：

In [ ]:
# ====== 故意制造报错，学习读报错 ======

# 1. IndexError: 索引越界
lst = [1, 2, 3]
try:
    lst[5]
except IndexError as e:
    print(f'IndexError: {e}')

In [ ]:
# 2. KeyError: 字典键不存在
d = {'a': 1, 'b': 2}
try:
    d['c']
except KeyError as e:
    print(f'KeyError: {e}')

In [ ]:
# 3. ValueError: 值错误
try:
    int('hello')
except ValueError as e:
    print(f'ValueError: {e}')

In [ ]:
# 4. TypeError: 类型错误
try:
    '2' + 2
except TypeError as e:
    print(f'TypeError: {e}')

In [ ]:
# 5. RuntimeError: 运行时错误（PyTorch中最常见）
# 最典型的：维度不匹配
import torch
import torch.nn as nn

linear = nn.Linear(10, 5)  # 期望输入维度为10
x = torch.randn(3, 8)      # 实际输入维度为8
try:
    y = linear(x)
except RuntimeError as e:
    print(f'RuntimeError: {e}')
    print()
    print('关键信息解读:')
    print('  - Mat1和Mat2维度不匹配')
    print('  - 最后一维: 8 vs 10')
    print('  - 意思是: 输入x的最后一个维度是8，但线性层期望10')
    print('  - 解决: 检查x.shape，确认最后一维是否正确')

### 报错信息阅读技巧

1. **从下往上读**——最后一行是错误类型和消息，最有信息量
2. **看Traceback**——它告诉你错误发生在哪一行、调用链是什么
3. **读最后一行两遍**——第一遍理解大意，第二遍找关键数字
4. **维度不匹配是最常见的错误**——养成习惯：每步都 `print(tensor.shape)`

### 1.2 调试方法演进

| 方法 | 优点 | 缺点 |
|------|------|------|
| print | 最简单 | 需要手动添加/删除 |
| pdb | 不需要改代码 | 交互式命令行，不太直观 |
| Jupyter调试器 | 可视化断点 | 只在notebook中有效 |
| VS Code调试器 | 最专业 | 需要配置 |

**建议**：在日常开发中，`print(tensor.shape)` 仍然是最快最实用的调试方法。

In [ ]:
# print调试示例：跟踪张量形状变化
import torch
import torch.nn as nn

class DebugCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)  # 输入1通道，输出16通道
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)  # 16→32
        self.pool = nn.MaxPool2d(2, 2)  # 尺寸减半
        self.fc = nn.Linear(32 * 8 * 8, 10)  # 全连接层
    
    def forward(self, x):
        # 每一步都打印形状
        print(f'输入: {x.shape}')
        
        x = self.pool(torch.relu(self.conv1(x)))
        print(f'conv1 + pool: {x.shape}')
        
        x = self.pool(torch.relu(self.conv2(x)))
        print(f'conv2 + pool: {x.shape}')
        
        x = x.view(x.size(0), -1)  # 展平
        print(f'flatten: {x.shape}')
        
        x = self.fc(x)
        print(f'fc输出: {x.shape}')
        return x

# 测试
model = DebugCNN()
x = torch.randn(4, 1, 32, 32)  # batch=4, channel=1, 32x32
y = model(x)

## 2. Git基础

Git是版本管理工具，在研究中它的作用是：

- **保存进度**：每次实验结果有改进就commit，随时可以回退
- **同步代码**：在服务器和笔记本之间同步
- **记录实验**：commit message就是实验记录


In [ ]:
# Git基本操作（在终端中执行，这里只展示命令）

# 初始化仓库
# git init

# 添加文件到暂存区
# git add .

# 提交（相当于保存进度）
# git commit -m "完成模块0的练习"

# 查看当前状态
# git status

# 查看历史
# git log --oneline

# 回退到之前的版本
# git checkout <commit-hash>

print('Git命令参考已就绪')

### Git commit message 规范

好的commit message就是实验记录：

```bash
# 好的例子
git commit -m "CNN分类器：2层conv+1层fc，准确率85%"
git commit -m "修改学习率：0.001→0.0001，loss下降更快"
git commit -m "增加数据增强：随机裁剪+频谱掩蔽，过拟合缓解"

# 坏的例子
git commit -m "update"
git commit -m "fix bug"
git commit -m "修改了一下"
```

**原则**：commit message要回答"这次改动做了什么，效果如何？"

## 3. 编程习惯

### 3.1 命名规范

| 类型 | 规范 | 例子 |
|------|------|------|
| 变量/函数 | snake_case | `sample_rate`, `compute_spectrum()` |
| 类 | PascalCase | `AudioDataset`, `CNNClassifier` |
| 常量 | UPPER_CASE | `SAMPLE_RATE`, `NUM_CLASSES` |
| 私有属性 | _前缀 | `self._internal_state` |

### 3.2 好名字 vs 坏名字

**变量名要回答"这个变量存储什么"，而不是"这个变量的类型是什么"。**

In [ ]:
# ====== 坏名字 vs 好名字 ======

# 坏名字
a = 16000          # 什么采样率？
b = 0.5            # 什么的振幅？
c = np.zeros(100)  # 什么的缓冲区？
tmp = wave[:500]   # 什么的临时变量？

# 好名字
sample_rate = 16000
amplitude = 0.5
output_buffer = np.zeros(100)
first_500_samples = wave[:500]

### 3.3 注释写法

**注释要解释"为什么"，而不是"做了什么"**——代码本身已经说明了"做了什么"。

In [ ]:
# ====== 坏注释 vs 好注释 ======

# 坏注释：重复代码
sr = 16000  # 设置采样率为16000
n = int(sr * dur)  # 计算样本数

# 好注释：解释"为什么"
sample_rate = 16000  # CI语音处理的常用采样率
num_samples = int(sample_rate * duration)  # int()截断而非四舍五入，避免最后一个样本溢出

### 3.4 代码格式化

使用 `black` 自动格式化代码，不再为格式争论浪费时间。

In [ ]:
# 安装black（在终端中执行）
# pip install black

# 使用black格式化当前notebook
# black notebook.ipynb
# 或者格式化整个项目
# black .

print('代码格式化工具介绍完毕')

## 4. 综合练习

**任务**：将前两次课的代码重构为模块化的项目，提交到Git仓库。

### 步骤

1. 创建项目目录结构：
   ```
   audio_tools/
   ├── __init__.py
   ├── signal.py          # Signal类
   ├── generators.py      # SineWave, NoisySignal
   └── visualization.py   # 画图工具函数
   ```
2. 将notebook中的代码整理到对应的.py文件中
3. 在notebook中用import引用自己的模块
4. 用Git提交，commit message写清楚


In [ ]:
# ====== 综合练习：模块化重构 ======

# 步骤1：创建目录结构
import os

project_dir = 'audio_tools'
os.makedirs(project_dir, exist_ok=True)

# 创建__init__.py（让Python识别这个目录为包）
with open(os.path.join(project_dir, '__init__.py'), 'w') as f:
    f.write('')

print(f'项目目录 {project_dir} 创建完成')

In [ ]:
# 步骤2：将Signal类写入signal.py
signal_code = '''
import numpy as np
import matplotlib.pyplot as plt

class Signal:
    """音频信号基类"""
    
    def __init__(self, waveform, sample_rate, label=''):
        self.waveform = waveform
        self.sample_rate = sample_rate
        self.label = label
    
    def plot(self):
        sr = self.sample_rate
        t = np.linspace(0, len(self.waveform)/sr, len(self.waveform), endpoint=False)
        plt.figure(figsize=(12, 4))
        plt.plot(t, self.waveform)
        plt.title(self.label)
        plt.show()
    
    def compute_spectrum(self):
        N = len(self.waveform)
        spectrum = np.abs(np.fft.rfft(self.waveform))
        freqs = np.fft.rfftfreq(N, 1/self.sample_rate)
        return freqs, spectrum
'''

with open(os.path.join(project_dir, 'signal.py'), 'w') as f:
    f.write(signal_code)
print('signal.py 写入完成')

In [ ]:
# 步骤2b：将SineWave和NoisySignal写入generators.py
generators_code = '''
import numpy as np
from .signal import Signal

class SineWave(Signal):
    """正弦波信号"""
    
    def __init__(self, frequency, amplitude=0.5, duration=1.0, sample_rate=16000):
        t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
        waveform = amplitude * np.sin(2 * np.pi * frequency * t)
        label = f'{frequency} Hz Sine Wave'
        super().__init__(waveform, sample_rate, label)
        self.frequency = frequency
        self.amplitude = amplitude
        self.duration = duration

class NoisySignal(Signal):
    """带噪信号"""
    
    def __init__(self, clean_waveform, sample_rate, noise_level=0.1, label=''):
        noise = np.random.randn(len(clean_waveform)) * noise_level
        noisy = clean_waveform + noise
        label = label or f'Noisy Signal (noise_level={noise_level})'
        super().__init__(noisy, sample_rate, label)
        self.clean_waveform = clean_waveform
        self.noise_level = noise_level
    
    def compute_snr(self):
        signal_power = np.mean(self.clean_waveform ** 2)
        noise = self.waveform - self.clean_waveform
        noise_power = np.mean(noise ** 2)
        if noise_power == 0:
            return float('inf')
        return 10 * np.log10(signal_power / noise_power)
'''

with open(os.path.join(project_dir, 'generators.py'), 'w') as f:
    f.write(generators_code)
print('generators.py 写入完成')

In [ ]:
# 步骤3：测试从模块导入
from audio_tools.signal import Signal
from audio_tools.generators import SineWave, NoisySignal

# 使用方式和之前完全一样
sine = SineWave(440)
print(f'频率: {sine.frequency} Hz')
print(f'采样率: {sine.sample_rate} Hz')

clean = SineWave(440)
noisy = NoisySignal(clean.waveform, clean.sample_rate, noise_level=0.3)
print(f'SNR: {noisy.compute_snr():.1f} dB')

## 本节要点

1. **读报错信息**：从下往上读，最后一行最有信息量；维度不匹配是最常见的bug
2. **调试方法**：`print(tensor.shape)` 是最快最实用的；Jupyter调试器和VS Code调试器更专业
3. **Git**：`init → add → commit` 是最基本的工作流；commit message要写清楚
4. **编程习惯**：好名字回答"存储什么"；注释解释"为什么"；用black自动格式化


---
返回目录：[README.md](../README.md)